In [ ]:
# 1) 설치
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 130.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00


In [ ]:
# 2) 모델 로드
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("모델 로드 완료")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

모델 로드 완료


In [ ]:
# 3) 채팅 함수
def chat_qwen(user_text, system_prompt="당신은 딸에게 대답하는 아버지 입니다."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_text},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature= 0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response

In [ ]:
# 4) 테스트
question = "아빠 동화책 읽어줘."
answer = chat_qwen(question)
print(answer)

물론이죠! 오늘은 "백설공주와 일곱 난쟁이"라는 동화를 읽어드리겠습니다.

从前，有一个美丽的公主叫做白雪公主。她非常善良和温柔，但她的继母皇后却非常嫉妒她。皇后有一面魔镜，可以告诉她谁是世界上最美丽的人。每次魔镜都会说白雪公主比任何人都美。

有一天，皇后问魔镜：“谁是最美的女人？” 魔镜回答：“白雪公主比任何人都美。”皇后非常生气，决定要除掉白雪公主。

于是，皇后命令一个猎人去杀死白雪公主。猎人不忍心杀她，就把她放走了。白雪公主逃到了森林里，在那里遇到了七个善良的小矮人。小矮人们欢迎她并让她住在他们的家里。

某天，皇后装扮成一位卖漂亮衣服的老妇人，来到了小矮人的家。她给了白雪公主一个有毒的苹果。白雪公主吃了苹果后立刻昏迷了过去。小矮人们发现了她，非常伤心。他们把她放在一棵大树下，每天轮流看守她。

有一天，一位英俊的王子经过这里，发现了躺在树下的白雪公主。他以为她已经死了，就吻了她。奇迹发生了
